# linear-affine-on-custom-tensor — faded example 3: Fill the input backward for the Linear

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `linear-affine-on-custom-tensor`. Running the beacon reports progress on the `Backprop: Linear affine on custom Tensor` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Linear affine on custom Tensor` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`linear-affine-on-custom-tensor`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "linear-affine-on-custom-tensor"
DD_SUBTOPIC = "Backprop: Linear affine on custom Tensor"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

For `out = x @ weight`, the gradient w.r.t. the input is `grad_out @ weight.T`. A `(B, out)` upstream gradient times `(out, in)` (the transposed weight) yields `(B, in)` — matching the input's shape.

## Faded exercise 3

Complete `linear_input_backward(grad_out, weight)`. It must return the input gradient of shape `(B, in)`. Fill in the matmul-with-transposed-weight expression.

**Fill in:** matmul of grad_out with the transposed weight to get the (B, in) input gradient

In [ ]:
import numpy as np
from dataclasses import dataclass

@dataclass
class Recipe:
    func: object
    args: tuple
    kwargs: dict
    parents: dict

class MiniTensor:
    def __init__(self, array, requires_grad=False, recipe=None):
        self.array = np.asarray(array, dtype=np.float64)
        self.requires_grad = requires_grad
        self.recipe = recipe

def linear_input_backward(grad_out, weight):
    gx = None  # TODO: matmul of grad_out with the transposed weight to get the (B, in) input gradient
    return gx


def _test():
    np.random.seed(9)
    weight = MiniTensor(np.random.randn(3, 4), requires_grad=True)
    grad_out = np.random.randn(5, 4)
    gx = linear_input_backward(grad_out, weight)
    assert gx.shape == (5, 3)
    # independent finite-difference truth on loss = sum(x @ weight)
    np.random.seed(10)
    x = np.random.randn(5, 3)
    eps = 1e-6
    num = np.zeros_like(x)
    for i in range(5):
        for j in range(3):
            xp = x.copy(); xp[i, j] += eps
            xm = x.copy(); xm[i, j] -= eps
            num[i, j] = ((xp @ weight.array).sum() - (xm @ weight.array).sum()) / (2 * eps)
    # for sum-loss grad_out is ones; recompute gx with that to compare to numeric
    gx_ones = np.ones((5, 4)) @ weight.array.T
    assert np.allclose(gx_ones, num, atol=1e-4)


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import numpy as np
from dataclasses import dataclass

@dataclass
class Recipe:
    func: object
    args: tuple
    kwargs: dict
    parents: dict

class MiniTensor:
    def __init__(self, array, requires_grad=False, recipe=None):
        self.array = np.asarray(array, dtype=np.float64)
        self.requires_grad = requires_grad
        self.recipe = recipe

def linear_input_backward(grad_out, weight):
    gx = grad_out @ weight.array.T
    return gx
```
</details>